# Cash Desk Video Check — evaluation results
Reports written by `eval/run_eval.py` on the 42 labelled videos (GPU server, live version v19),
plus per-frame output of the pipeline for videos X-26, X-20 and X-21 (`timeline.json`).

In [ ]:
import glob, json
import pandas as pd
import matplotlib.pyplot as plt

def report(tag):
    path = sorted(glob.glob(f"../out/eval/*-{tag}.json"))[-1]
    return json.load(open(path))

def score(rep):
    v = rep["videos"]
    return (sum(x["gt"] and x["pred"] for x in v), sum(x["gt"] for x in v),
            sum(not x["gt"] and x["pred"] for x in v), sum(not x["gt"] for x in v))

live = report("v19-orig")
caught, violations, false_alarms, clean = score(live)
o = live["overall"]
pd.Series({
    "Violation videos caught": f"{caught} / {violations}",
    "False alarms on clean videos": f"{false_alarms} / {clean}",
    "Timecode within 2 s of the label": f"{o['timecode_within_2s']:.0%}",
    "Correct violation type": f"{o['type_accuracy_confident']:.0%}",
    "Mean processing time per video": f"{o['runtime_mean']:.1f} s",
}, name="live service (v19)").to_frame()

In [ ]:
# Every change, measured on all 42 videos (version log from README.md)
steps = pd.DataFrame([
    ("Rules only", 11, 0), ("GPU server", 12, 0), ("+ vision model", 15, 2), ("+ coat/bag check", 15, 2),
    ("coat/bag tightened", 15, 0), ("+ entry rule", 15, 4), ("entry rule dropped", 15, 0),
    ("camera tests", 15, 0), ("honest cup check", 13, 0), ("real timestamps", 13, 0),
], columns=["step", "caught (of 22)", "false alarms (of 20)"]).set_index("step")
ax = steps.plot.bar(figsize=(11, 4.2), rot=30)
ax.set_title("Caught violation videos and false alarms after each change")
plt.tight_layout(); plt.show()

In [ ]:
tl = json.load(open("timeline.json"))
x = tl["X-26"]
t_up, t_empty = x["departures"][0]
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.step(x["times"], x["people"], where="post")
ax.axvline(t_up, color="red", linestyle="--", label=f"teller stands up ({t_up:.0f} s) = reported time")
ax.axvline(t_empty, color="gray", linestyle="--", label=f"room empty ({t_empty:.0f} s); cash still on desk")
ax.set_yticks([0, 1, 2]); ax.set_xlabel("seconds into the video"); ax.set_ylabel("people in the room")
ax.set_title(f"X-26: {x['events'][0][0]} reported at {x['events'][0][1]:.0f} s")
ax.legend(loc="lower left"); plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
for name, label in (("X-20", "X-20: door left open"), ("X-21", "X-21: clean video")):
    s = tl[name]["safe_samples"]
    ax.plot([p[0] for p in s], [p[1] for p in s], "-o", label=label)
ax.axhline(0.6, color="gray", linestyle="--", label="threshold")
ax.set_xlabel("seconds into the video"); ax.set_ylabel('P("the safe is open")')
ax.set_title("Reported only if open for 6 s or more: X-20 reported, X-21 (one open sample) not")
ax.legend(loc="center right"); plt.tight_layout(); plt.show()

In [ ]:
tests = {"Original": "v19-orig", "Mirrored": "v16-mirror", "Camera tilted": "v16-tilt",
         "Zoomed in 15%": "v19-zoom", "Laptop hidden": "v16-nolaptop2", "Lower quality": "v16-lowq"}
rows = pd.DataFrame({name: score(report(tag)) for name, tag in tests.items()},
                    index=["caught", "violations", "false alarms", "clean"]).T
ax = rows[["caught", "false alarms"]].plot.barh(figsize=(11, 4.2))
ax.invert_yaxis()
ax.set_title("Same 42 videos, distorted as if filmed by another camera (of 22 violations / 20 clean)")
plt.tight_layout(); plt.show()
rows

In [ ]:
labels = pd.read_csv("../data/labels.csv").set_index("video")
runs = pd.DataFrame(live["videos"]).set_index("video")
runs["length_s"] = labels["duration_s"]
ax = runs.plot.scatter(x="length_s", y="runtime", figsize=(11, 4.2))
ax.axhline(300, color="red", linestyle="--", label="time limit: 300 s per video")
ax.set_xlabel("video length, s"); ax.set_ylabel("processing time, s"); ax.set_ylim(0, 320); ax.legend()
ax.set_title(f"All {len(runs)} videos: mean {runs['runtime'].mean():.1f} s, max {runs['runtime'].max():.1f} s")
plt.tight_layout(); plt.show()